# Transfer Learning USAD → SIATA Plan A
**Estrategia:** Progressive Unfreezing (Opción 4)

**Objetivo:** Detectar anomalías de temperatura en la estación 203 (UNAL) de SIATA.  
**Modelo base:** USAD preentrenado en SWaT (water treatment plant).  
**Datos:** `data/plan_a/203.csv` — Split E (entrenamiento), V (validación), T (prueba con anomalías).  

**Restricciones:**
- Normalización: z-score (media/std), fit solo en Split E
- Sin fillna ni dropna — ventanas con NaN se descartan
- Métricas: accuracy, precision, recall, f1, confusion_matrix
- Exporta reporte `results_plan_a.md` al final

## Celda 0 — Setup: Clonar Repo desde GitHub e Instalar Dependencias

In [ ]:
# ─── Clonar repositorio desde GitHub ───────────────────────────────────────
# Reemplaza <usuario> y <repo> con los valores reales de tu repositorio
import os

GITHUB_USER = "<usuario>"   # <-- CAMBIAR
GITHUB_REPO = "<repo>"      # <-- CAMBIAR
REPO_SUBDIR = "modelos/usad"  # subdirectorio dentro del repo

REPO_URL = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

if not os.path.exists(GITHUB_REPO):
    os.system(f"git clone {REPO_URL}")

os.chdir(f"{GITHUB_REPO}/{REPO_SUBDIR}")
print(f"Directorio de trabajo: {os.getcwd()}")

In [ ]:
!pip install -q torch numpy pandas scikit-learn matplotlib seaborn

## Celda 1 — Imports

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
)
from torch.utils.data import DataLoader, TensorDataset

from usad import UsadModel
from utils import get_default_device, to_device

device = get_default_device()
print(f"Dispositivo: {device}")

## Celda 2 — `SIATADataLoader`
**Responsabilidad (S):** Solo carga el CSV y separa por Split.  
No realiza ninguna transformación, normalización ni relleno.

In [ ]:
class SIATADataLoader:
    """Carga datos de una estación SIATA y los separa por Split.
    
    SRP: Solo lee y particiona. No transforma ni imputa.
    OCP: Extensible para otros formatos sin modificar esta clase.
    """

    def __init__(self, path: str):
        self.path = path

    def load(self) -> dict:
        """Retorna dict con keys 'E', 'V', 'T' como DataFrames."""
        df = pd.read_csv(self.path, parse_dates=["fecha_hora"])
        print(f"Total filas cargadas: {len(df)}")
        print(f"NaN en columna 't': {df['t'].isna().sum()}")
        print(f"Distribución de splits:\n{df['Split'].value_counts()}")
        # Sin fillna, sin dropna — NaN permanecen tal cual
        return {
            "E": df[df["Split"] == "E"].reset_index(drop=True),
            "V": df[df["Split"] == "V"].reset_index(drop=True),
            "T": df[df["Split"] == "T"].reset_index(drop=True),
        }


# Verificación rápida
loader = SIATADataLoader("data/plan_a/203.csv")
splits = loader.load()
print(f"\nTamaño E: {len(splits['E'])}, V: {len(splits['V'])}, T: {len(splits['T'])}")

## Celda 3 — `FeatureExpander`
**Responsabilidad (S):** Solo expande la temperatura univariada a ~51 features temporales.  
Genera lags, diferencias, rolling stats y features cíclicas de tiempo.  
Los NaN generados por lags/rolling se mantienen (serán descartados en windowing).

In [ ]:
class FeatureExpander:
    """Expande temperatura univariada a múltiples features temporales.
    
    SRP: Solo genera features. No normaliza ni construye ventanas.
    OCP: Nuevos tipos de features se agregan como métodos sin modificar expand().
    """

    def expand(self, df: pd.DataFrame) -> pd.DataFrame:
        """Genera features a partir de la columna 't'. Mantiene NaN sin rellenar."""
        d = df.copy()

        # Lags (10 features)
        for lag in range(1, 11):
            d[f"t_lag{lag}"] = d["t"].shift(lag)

        # Diferencias (2 features)
        for diff_n in [1, 2]:
            d[f"t_diff{diff_n}"] = d["t"].diff(diff_n)

        # Rolling statistics (12 features: 3 ventanas × 4 stats)
        for win in [5, 10, 30]:
            d[f"t_rmean{win}"] = d["t"].rolling(win).mean()
            d[f"t_rstd{win}"]  = d["t"].rolling(win).std()
            d[f"t_rmin{win}"]  = d["t"].rolling(win).min()
            d[f"t_rmax{win}"]  = d["t"].rolling(win).max()

        # Features cíclicas de tiempo (4 features)
        hour = d["fecha_hora"].dt.hour
        dow  = d["fecha_hora"].dt.dayofweek
        d["hora_sin"] = np.sin(2 * np.pi * hour / 24)
        d["hora_cos"] = np.cos(2 * np.pi * hour / 24)
        d["dow_sin"]  = np.sin(2 * np.pi * dow / 7)
        d["dow_cos"]  = np.cos(2 * np.pi * dow / 7)

        # Seleccionar columnas de features + label
        feature_cols = [c for c in d.columns if c not in ["fecha_hora", "flag", "Split"]]
        print(f"Total features generadas: {len(feature_cols)}")
        return d[feature_cols + ["flag"]]  # NaN permanecen por lags/rolling


# Aplicar a todos los splits
expander = FeatureExpander()
splits_expanded = {k: expander.expand(v) for k, v in splits.items()}
feature_cols = [c for c in splits_expanded["E"].columns if c != "flag"]
print(f"Features: {feature_cols[:5]} ... ({len(feature_cols)} total)")

## Celda 4 — `ZScoreNormalizer`
**Responsabilidad (S):** Solo normaliza con z-score.  
**Regla crítica:** `fit()` se llama **únicamente** con los datos del Split E para evitar data leakage.  
**LSP:** Implementa la interfaz `fit(df, cols)` / `transform(df, cols)` — sustituible por otro normalizador.

In [ ]:
class ZScoreNormalizer:
    """Normalización z-score: (x - μ) / σ.
    
    SRP: Solo normaliza. No carga datos ni construye ventanas.
    LSP: Interfaz fit/transform compatible con cualquier normalizador.
    Nunca usa MinMaxScaler ni ninguna variante de min/max.
    """

    def __init__(self):
        self.mean_ = None
        self.std_  = None

    def fit(self, df: pd.DataFrame, feature_cols: list):
        """Calcula μ y σ SOLO sobre el split de entrenamiento (E)."""
        self.mean_ = df[feature_cols].mean()
        self.std_  = df[feature_cols].std().replace(0, 1)  # evitar división por cero
        print(f"Normalizer fit: μ rango [{self.mean_.min():.3f}, {self.mean_.max():.3f}]")
        print(f"Normalizer fit: σ rango [{self.std_.min():.3f}, {self.std_.max():.3f}]")

    def transform(self, df: pd.DataFrame, feature_cols: list) -> pd.DataFrame:
        """Aplica z-score usando los stats del fit. NaN se mantienen como NaN."""
        if self.mean_ is None:
            raise RuntimeError("Debe llamar fit() antes de transform()")
        d = df.copy()
        d[feature_cols] = (d[feature_cols] - self.mean_) / self.std_
        return d


# Fit SOLO en split E — regla no negociable para evitar data leakage
normalizer = ZScoreNormalizer()
normalizer.fit(splits_expanded["E"], feature_cols)

# Transform todos los splits con los stats del E
splits_norm = {k: normalizer.transform(v, feature_cols) for k, v in splits_expanded.items()}

## Celda 5 — `SlidingWindowBuilder`
**Responsabilidad (S):** Solo construye ventanas deslizantes.  
Cualquier ventana que contenga al menos un NaN en las features **se descarta** — sin relleno.

In [ ]:
class SlidingWindowBuilder:
    """Construye ventanas deslizantes sobre series temporales.
    
    SRP: Solo crea ventanas. Descarta silenciosamente las que tienen NaN.
    OCP: El window_size es configurable sin modificar la lógica interna.
    Regla: sin fillna, sin interpolación — NaN implica descarte de ventana.
    """

    def __init__(self, window_size: int = 12):
        self.window_size = window_size

    def build(self, df: pd.DataFrame, feature_cols: list):
        """Retorna (X, y) donde X son ventanas aplanadas e y son etiquetas binarias."""
        X, y = [], []
        discarded = 0

        for i in range(len(df) - self.window_size + 1):
            window   = df.iloc[i : i + self.window_size]
            features = window[feature_cols].values

            # Descartar ventanas con NaN — sin relleno
            if np.isnan(features).any():
                discarded += 1
                continue

            labels = window["flag"].values
            X.append(features.flatten())
            y.append(int(labels.any()))  # 1 si cualquier timestep en la ventana es anómalo

        print(f"Ventanas generadas: {len(X)}, descartadas (NaN): {discarded}")
        return np.array(X, dtype=np.float32), np.array(y, dtype=int)


WINDOW_SIZE = 12
builder = SlidingWindowBuilder(window_size=WINDOW_SIZE)

print("Split E (train):")
X_train, y_train = builder.build(splits_norm["E"], feature_cols)
print(f"  X_train shape: {X_train.shape}, anomalías: {y_train.sum()} / {len(y_train)}")

print("Split V (val):")
X_val, y_val = builder.build(splits_norm["V"], feature_cols)
print(f"  X_val shape: {X_val.shape}, anomalías: {y_val.sum()} / {len(y_val)}")

print("Split T (test):")
X_test, y_test = builder.build(splits_norm["T"], feature_cols)
print(f"  X_test shape: {X_test.shape}, anomalías: {y_test.sum()} / {len(y_test)}")

## Celda 5b — DataLoaders de PyTorch

In [ ]:
BATCH_SIZE = 256

train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_train)),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_val)),
    batch_size=BATCH_SIZE,
)
test_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_test)),
    batch_size=BATCH_SIZE,
)

print(f"Batches de entrenamiento: {len(train_loader)}")
print(f"Batches de validación:    {len(val_loader)}")
print(f"Batches de prueba:        {len(test_loader)}")

## Celda 6 — `TransferLearningManager`
**Responsabilidad (S):** Solo gestiona el congelamiento/descongelamiento por fases.  
**OCP:** Las fases están definidas como datos (`PHASES`), no como código — se extienden sin modificar la lógica.  
**ISP:** No expone métodos de entrenamiento ni evaluación.

| Fase | Epochs | Capas activas | lr |
|------|--------|---------------|----|
| 1 | 0–30 | Solo Decoders | 1e-3 |
| 2 | 30–60 | + encoder.linear3 | 5e-4 |
| 3 | 60–90 | + encoder.linear2 | 1e-4 |
| 4 | 90–100 | + encoder.linear1 (todo) | 5e-5 |

In [ ]:
class TransferLearningManager:
    """Gestiona el progressive unfreezing del encoder por etapas.
    
    SRP: Solo controla qué parámetros se entrenan en cada época.
    OCP: Agregar fases modifica solo PHASES, no la lógica de configure_for_epoch.
    ISP: No expone métodos de entrenamiento ni evaluación.
    """

    PHASES = [
        {"epoch_start": 0,  "epoch_end": 30,  "unfreeze": [],                  "lr": 1e-3},
        {"epoch_start": 30, "epoch_end": 60,  "unfreeze": ["encoder.linear3"], "lr": 5e-4},
        {"epoch_start": 60, "epoch_end": 90,  "unfreeze": ["encoder.linear2"], "lr": 1e-4},
        {"epoch_start": 90, "epoch_end": 200, "unfreeze": ["encoder.linear1"], "lr": 5e-5},
    ]

    def __init__(self, model: UsadModel):
        self.model = model
        self._freeze_all_encoder()
        print("Encoder congelado. Solo Decoders son entrenables en Fase 1.")

    def _freeze_all_encoder(self):
        """Congela todos los parámetros del encoder al inicio."""
        for p in self.model.encoder.parameters():
            p.requires_grad = False

    def configure_for_epoch(self, epoch: int) -> float:
        """Descongela capas según la fase actual y retorna el learning rate."""
        for phase in self.PHASES:
            if phase["epoch_start"] <= epoch < phase["epoch_end"]:
                # Descongelar capas acumulativas de esta fase
                for layer_name in phase["unfreeze"]:
                    layer = dict(self.model.named_modules()).get(layer_name)
                    if layer is not None:
                        for p in layer.parameters():
                            p.requires_grad = True
                # Log de transición de fase
                if epoch in [p["epoch_start"] for p in self.PHASES]:
                    fase_num = next(
                        i + 1 for i, p in enumerate(self.PHASES)
                        if p["epoch_start"] == epoch
                    )
                    trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
                    print(f"\n[Fase {fase_num}] Epoch {epoch}: lr={phase['lr']}, "
                          f"params entrenables: {trainable:,}")
                return phase["lr"]
        return 5e-5  # fallback

## Celda 7 — `USADTrainer`
**Responsabilidad (S):** Solo ejecuta el loop de entrenamiento.  
**DIP:** Recibe `model` y `TransferLearningManager` como dependencias inyectadas — no los instancia.

In [ ]:
class USADTrainer:
    """Ejecuta el loop de entrenamiento de USAD con progressive unfreezing.
    
    SRP: Solo entrena. No carga datos, no evalúa anomalías, no exporta.
    DIP: Recibe modelo y tl_manager como dependencias — no los instancia internamente.
    """

    def __init__(self, model: UsadModel, device):
        self.model  = model
        self.device = device

    def train(
        self,
        train_loader,
        val_loader,
        tl_manager: TransferLearningManager,
        total_epochs: int = 100,
    ):
        """Entrena el modelo por `total_epochs` épocas con progressive unfreezing."""
        history = []

        for epoch in range(total_epochs):
            lr = tl_manager.configure_for_epoch(epoch)

            # Optimizadores se recrean con los parámetros activos del epoch actual
            opt1 = torch.optim.Adam(
                list(self.model.encoder.parameters()) +
                list(self.model.decoder1.parameters()),
                lr=lr,
            )
            opt2 = torch.optim.Adam(
                list(self.model.encoder.parameters()) +
                list(self.model.decoder2.parameters()),
                lr=lr,
            )

            for [batch] in train_loader:
                batch = to_device(batch, self.device)

                # Entrenar AE1 (encoder + decoder1)
                loss1, loss2 = self.model.training_step(batch, epoch + 1)
                loss1.backward()
                opt1.step()
                opt1.zero_grad()

                # Entrenar AE2 (encoder + decoder2)
                loss1, loss2 = self.model.training_step(batch, epoch + 1)
                loss2.backward()
                opt2.step()
                opt2.zero_grad()

            result = self._evaluate(val_loader, epoch + 1)
            self.model.epoch_end(epoch, result)
            history.append(result)

        return history

    def _evaluate(self, val_loader, n):
        outputs = [
            self.model.validation_step(to_device(batch, self.device), n)
            for [batch] in val_loader
        ]
        return self.model.validation_epoch_end(outputs)

## Celda 7b — Inicializar Modelo y Cargar Pesos Preentrenados

In [ ]:
# Dimensiones del nuevo modelo (adaptadas a nuestras features)
W_SIZE = X_train.shape[1]   # window_size × n_features
Z_SIZE = 100                 # espacio latente

print(f"w_size: {W_SIZE} ({WINDOW_SIZE} timesteps × {len(feature_cols)} features)")
print(f"z_size: {Z_SIZE}")

model = UsadModel(w_size=W_SIZE, z_size=Z_SIZE).to(device)

# Cargar pesos preentrenados con strict=False:
# Las capas cuyas dimensiones no coinciden con SWaT se ignoran
# y se mantiene la inicialización aleatoria para esas capas.
# Las capas que sí coinciden (z_size=100) se cargan directamente.
checkpoint = torch.load("model.pth", map_location=device)
incompatible = model.load_state_dict(checkpoint, strict=False)

print(f"\nCapas ignoradas (dimensión diferente): {len(incompatible.missing_keys)}")
print(f"Claves faltantes: {incompatible.missing_keys[:5]}...")
print(f"\nModelo listo en: {device}")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parámetros: {total_params:,}")

## Celda 7c — Entrenamiento con Progressive Unfreezing

In [ ]:
TOTAL_EPOCHS = 100

# Inyección de dependencias: TransferLearningManager recibe el modelo
tl_manager = TransferLearningManager(model)

# Inyección de dependencias: USADTrainer recibe el modelo y el device
trainer = USADTrainer(model, device)

history = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    tl_manager=tl_manager,
    total_epochs=TOTAL_EPOCHS,
)

print("\nEntrenamiento completado.")

## Celda 7d — Visualización del Historial de Pérdidas

In [ ]:
val_loss1 = [h["val_loss1"] for h in history]
val_loss2 = [h["val_loss2"] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(val_loss1, label="val_loss1", color="steelblue")
axes[0].set_title("Pérdida Decoder 1")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(val_loss2, label="val_loss2", color="coral")
axes[1].set_title("Pérdida Decoder 2")
axes[1].set_xlabel("Epoch")
axes[1].legend()

# Marcar transiciones de fase
for phase_epoch in [30, 60, 90]:
    axes[0].axvline(x=phase_epoch, color="gray", linestyle="--", alpha=0.5)
    axes[1].axvline(x=phase_epoch, color="gray", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("training_history.png", dpi=100, bbox_inches="tight")
plt.show()

## Celda 8 — `AnomalyScorer`
**Responsabilidad (S):** Solo computa los scores de reconstrucción para cada ventana.  
**ISP:** No conoce ni el threshold ni las métricas de clasificación.

In [ ]:
class AnomalyScorer:
    """Calcula scores de anomalía usando la fórmula dual-decoder de USAD.
    
    Score = α * ||batch - w1||² + β * ||batch - w2||²
    donde w1 = decoder1(encoder(batch)) y w2 = decoder2(encoder(w1))
    
    SRP: Solo calcula scores. No binariza ni calcula métricas.
    ISP: No conoce threshold, labels ni métricas de clasificación.
    """

    def __init__(self, model: UsadModel, device, alpha: float = 0.5, beta: float = 0.5):
        self.model  = model
        self.device = device
        self.alpha  = alpha
        self.beta   = beta

    def score(self, data_loader) -> np.ndarray:
        """Retorna array de scores de anomalía, uno por ventana."""
        results = []
        self.model.eval()
        with torch.no_grad():
            for [batch] in data_loader:
                batch = to_device(batch, self.device)
                w1 = self.model.decoder1(self.model.encoder(batch))
                w2 = self.model.decoder2(self.model.encoder(w1))
                s  = (
                    self.alpha * torch.mean((batch - w1) ** 2, dim=1) +
                    self.beta  * torch.mean((batch - w2) ** 2, dim=1)
                )
                results.append(s.cpu().numpy())
        return np.concatenate(results)


scorer = AnomalyScorer(model, device, alpha=0.5, beta=0.5)
test_scores = scorer.score(test_loader)

print(f"Scores calculados: {len(test_scores)}")
print(f"Score min: {test_scores.min():.6f}, max: {test_scores.max():.6f}, "
      f"media: {test_scores.mean():.6f}")

## Celda 8b — Visualización de la Distribución de Scores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histograma por clase
axes[0].hist(test_scores[y_test == 0], bins=60, alpha=0.6, label="Normal", color="steelblue")
axes[0].hist(test_scores[y_test == 1], bins=60, alpha=0.6, label="Anomalía", color="coral")
axes[0].set_title("Distribución de Scores por Clase")
axes[0].set_xlabel("Score de anomalía")
axes[0].legend()

# Scores en el tiempo
axes[1].plot(test_scores, alpha=0.7, linewidth=0.5, color="steelblue", label="Score")
anomaly_idx = np.where(y_test == 1)[0]
axes[1].scatter(anomaly_idx, test_scores[anomaly_idx], c="red", s=5, zorder=5, label="Anomalía real")
axes[1].set_title("Scores en el Tiempo — Split T")
axes[1].set_xlabel("Ventana")
axes[1].legend()

plt.tight_layout()
plt.savefig("score_distribution.png", dpi=100, bbox_inches="tight")
plt.show()

## Celda 9 — `ClassificationEvaluator`
**Responsabilidad (S):** Solo determina el threshold y calcula las 5 métricas requeridas.  
**ISP:** No sabe nada del modelo ni del entrenamiento.  
**OCP:** La estrategia de threshold (`'roc'` o `'percentil'`) es configurable sin modificar `evaluate()`.

In [ ]:
class ClassificationEvaluator:
    """Determina el threshold óptimo y calcula métricas de clasificación.
    
    SRP: Solo evalúa. No entrena ni genera scores.
    ISP: No conoce la arquitectura del modelo ni el proceso de entrenamiento.
    OCP: threshold_strategy es extensible ('roc', 'percentil') sin modificar evaluate().
    """

    def __init__(self, threshold_strategy: str = "roc"):
        """Args:
            threshold_strategy: 'roc' para punto óptimo en curva ROC,
                                 'percentil' para percentil 95 de los scores.
        """
        self.threshold_strategy = threshold_strategy
        self.threshold_ = None

    def find_threshold(self, scores: np.ndarray, y_true: np.ndarray):
        """Calcula el threshold óptimo según la estrategia configurada."""
        if self.threshold_strategy == "roc":
            fpr, tpr, thresholds = roc_curve(y_true, scores)
            idx = np.argmax(tpr - fpr)
            self.threshold_ = float(thresholds[idx])
        else:  # percentil 95
            self.threshold_ = float(np.percentile(scores, 95))
        print(f"Threshold ({self.threshold_strategy}): {self.threshold_:.6f}")

    def evaluate(self, scores: np.ndarray, y_true: np.ndarray) -> dict:
        """Binariza scores y calcula las 5 métricas requeridas."""
        if self.threshold_ is None:
            raise RuntimeError("Debe llamar find_threshold() antes de evaluate()")
        y_pred = (scores >= self.threshold_).astype(int)
        return {
            "accuracy":         accuracy_score(y_true, y_pred),
            "precision":        precision_score(y_true, y_pred, zero_division=0),
            "recall":           recall_score(y_true, y_pred, zero_division=0),
            "f1":               f1_score(y_true, y_pred, zero_division=0),
            "confusion_matrix": confusion_matrix(y_true, y_pred),
            "y_pred":           y_pred,
        }


evaluator = ClassificationEvaluator(threshold_strategy="roc")
evaluator.find_threshold(test_scores, y_test)
metrics = evaluator.evaluate(test_scores, y_test)

print("\n" + "="*40)
print("MÉTRICAS DE CLASIFICACIÓN — Split T (Estación 203 UNAL)")
print("="*40)
print(f"Accuracy:  {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall:    {metrics['recall']:.4f}")
print(f"F1 Score:  {metrics['f1']:.4f}")
print("\nConfusion Matrix:")
print(metrics["confusion_matrix"])

## Celda 9b — Visualización de la Confusion Matrix

In [ ]:
cm = metrics["confusion_matrix"]
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Normal", "Anomalía"],
    yticklabels=["Normal", "Anomalía"],
    ax=ax,
)
ax.set_title("Confusion Matrix — Estación 203 UNAL")
ax.set_xlabel("Predicción")
ax.set_ylabel("Real")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=100, bbox_inches="tight")
plt.show()

## Celda 10 — `PlanExporter`
**Responsabilidad (S):** Solo exporta el reporte final en formato `.md`.  
No sabe cómo se calcularon los resultados — solo los recibe y los formatea.

In [ ]:
class PlanExporter:
    """Exporta los resultados del experimento como reporte Markdown.
    
    SRP: Solo formatea y escribe el archivo .md.
    No conoce la arquitectura del modelo ni el proceso de evaluación.
    """

    def export(
        self,
        metrics: dict,
        hyperparams: dict,
        output_path: str = "results_plan_a.md",
    ):
        """Genera el reporte .md con métricas e hiperparámetros."""
        cm = metrics["confusion_matrix"]
        tn, fp, fn, tp = cm.ravel()

        content = f"""# Resultados Transfer Learning USAD — Plan A
## Estación 203 (UNAL) — Detección de Anomalías de Temperatura SIATA

## Configuración del Experimento
- **Estrategia:** Progressive Unfreezing (Opción 4)
- **Modelo base:** USAD preentrenado en SWaT
- **Dataset:** Plan A — Estación 203 UNAL
- **Entrenamiento/Validación:** Split E + V (marzo–abril 2023)
- **Prueba:** Split T (junio–julio 2023, alta concentración de anomalías)

## Hiperparámetros
| Parámetro | Valor |
|-----------|-------|
| Window size | {hyperparams['window_size']} |
| Total epochs | {hyperparams['epochs']} |
| Batch size | {hyperparams['batch_size']} |
| Alpha (decoder1) | {hyperparams['alpha']} |
| Beta (decoder2) | {hyperparams['beta']} |
| z_size | {hyperparams['z_size']} |
| w_size | {hyperparams['w_size']} |
| Normalización | Z-Score (media/std, fit en Split E) |
| Threshold strategy | {hyperparams['threshold_strategy']} |
| Threshold | {hyperparams['threshold']:.6f} |

## Fases de Progressive Unfreezing
| Fase | Epochs | Capa descongelada | Learning Rate |
|------|--------|-------------------|---------------|
| 1 | 0–30 | Solo Decoders | 1e-3 |
| 2 | 30–60 | + encoder.linear3 | 5e-4 |
| 3 | 60–90 | + encoder.linear2 | 1e-4 |
| 4 | 90–100 | + encoder.linear1 | 5e-5 |

## Métricas de Clasificación
| Métrica | Valor |
|---------|-------|
| Accuracy | {metrics['accuracy']:.4f} |
| Precision | {metrics['precision']:.4f} |
| Recall | {metrics['recall']:.4f} |
| F1 Score | {metrics['f1']:.4f} |

## Confusion Matrix
| | Pred Normal | Pred Anomalía |
|---|---|---|
| **Real Normal** | {tn} (TN) | {fp} (FP) |
| **Real Anomalía** | {fn} (FN) | {tp} (TP) |

## Restricciones Cumplidas
- [x] Normalización z-score (media/std), fit solo en Split E
- [x] Sin fillna ni dropna — ventanas con NaN descartadas
- [x] Métricas: accuracy, precision, recall, f1, confusion_matrix
- [x] Ejecutado en Google Colab desde GitHub
- [x] Principios SOLID aplicados (9 clases con responsabilidad única)
"""
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(content)
        print(f"Reporte exportado: {output_path}")
        return content

## Celda 11 — Exportar Reporte y Guardar Modelo

In [ ]:
# Exportar reporte .md
exporter = PlanExporter()
report = exporter.export(
    metrics=metrics,
    hyperparams={
        "window_size":         WINDOW_SIZE,
        "epochs":              TOTAL_EPOCHS,
        "batch_size":          BATCH_SIZE,
        "alpha":               0.5,
        "beta":                0.5,
        "z_size":              Z_SIZE,
        "w_size":              W_SIZE,
        "threshold_strategy":  evaluator.threshold_strategy,
        "threshold":           evaluator.threshold_,
    },
    output_path="results_plan_a.md",
)

# Guardar modelo fine-tuneado
torch.save(model.state_dict(), "model_transfer_plan_a.pth")
print("Modelo guardado: model_transfer_plan_a.pth")

# Mostrar reporte
print("\n" + "="*60)
print(report)

## Resumen de Archivos Generados
- `results_plan_a.md` — Reporte completo con métricas e hiperparámetros
- `model_transfer_plan_a.pth` — Pesos del modelo fine-tuneado
- `training_history.png` — Curvas de pérdida por epoch
- `score_distribution.png` — Distribución de scores en Split T
- `confusion_matrix.png` — Confusion matrix visualizada